# Byte Pair Encoding (BPE)

This notebook implements Byte Pair Encoding (BPE) tokenization and demonstrates how to build a vocabulary, encode text, and use the resulting tokens in simple language modeling experiments.

Sections:
- Data preparation
- BPE merges and vocabulary construction
- Tokenization examples
- Model training and evaluation


Notes: This overview cell summarizes the notebook's goals and sections. Use this as a high-level reference for what the notebook covers (data prep, BPE merges, tokenization examples, and model experiments).

In [2]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.14.0


Notes: Ensures `tiktoken` is installed and prints the package version. Run this cell first to verify the tokenizer runtime dependency.

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")


Notes: Initialize the GPT-2 byte-pair encoding tokenizer via `tiktoken.get_encoding("gpt2")`. This tokenizer will be used for encoding/decoding examples and dataset preparation.

In [4]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


Notes: Demonstrates encoding a short text string into token IDs using the tokenizer. Shows how to pass `allowed_special` for special tokens like `<|endoftext|>`.

In [5]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


Notes: Decodes the list of token IDs back to a human-readable string to demonstrate round-trip consistency between `encode` and `decode`.

In [8]:
text = (
    "Akwirwier"
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[33901, 86, 343, 86, 959]


Notes: Encodes a short (likely OOV) string to show how the tokenizer handles unknown or unusual character sequences. Useful to see tokenization granularity.

In [9]:
strings = tokenizer.decode(integers)
print(strings)

Akwirwier


Notes: Demonstrates decoding the previous example back to a string, confirming that even odd inputs can be reconstructed from token IDs.

In [10]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


Notes: Loads the dataset from `the-verdict.txt` and encodes the entire text using the tokenizer; `enc_text` is the token ID list used throughout the notebook.

In [11]:
enc_sample = enc_text[50:]

Notes: Creates a sample slice of the encoded text (`enc_sample`) for small-scale demonstrations and toy examples (e.g., showing context and next-token predictions).

In [12]:
context_size = 4         #1
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


Notes: Sets a `context_size` and creates example input (`x`) and label (`y`) sequences from `enc_sample` to demonstrate how context-target pairs are constructed for language modeling.

In [14]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


Notes: Iterates over increasing context lengths to show how progressively larger contexts map to their next-token target. Useful for understanding autoregressive training examples.

In [15]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


Notes: Decode the token `context` and `desired` token to readable strings so you can see the exact substrings the tokenizer produces for each context-target pair.

## Vocabulary & BPE Merges

The next code cell constructs the vocabulary and the BPE merges used to tokenize the dataset. It loads text, counts symbol pairs, performs merges iteratively, and builds the final token-to-index mapping used by the model.

Notes: Section header introducing vocabulary building and byte-pair merges; the following cells will build datasets and demonstrate tokenization and embedding usage.

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)    #1

        for i in range(0, len(token_ids) - max_length, stride):     #2
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self): #3
        return len(self.input_ids)

    def __getitem__(self, idx): #4
        return self.input_ids[idx], self.target_ids[idx]

Notes: Defines `GPTDatasetV1`, a PyTorch `Dataset` that turns encoded token IDs into sliding-window `input` and `target` tensors for autoregressive training. Key parameters: `max_length` and `stride` control window size and overlap.

In [18]:
gpt_obj = GPTDatasetV1(raw_text, tokenizer, max_length=8, stride=4)

Notes: Instantiate the `GPTDatasetV1` with the full `raw_text`, using `max_length=8` and `stride=4` to prepare example input/target pairs for inspection or training.

In [22]:
print(gpt_obj.input_ids)
print(gpt_obj.target_ids)

[tensor([  40,  367, 2885, 1464, 1807, 3619,  402,  271]), tensor([ 1807,  3619,   402,   271, 10899,  2138,   257,  7026]), tensor([10899,  2138,   257,  7026, 15632,   438,  2016,   257]), tensor([15632,   438,  2016,   257,   922,  5891,  1576,   438]), tensor([ 922, 5891, 1576,  438,  568,  340,  373,  645]), tensor([ 568,  340,  373,  645, 1049, 5975,  284,  502]), tensor([1049, 5975,  284,  502,  284, 3285,  326,   11]), tensor([ 284, 3285,  326,   11,  287,  262, 6001,  286]), tensor([  287,   262,  6001,   286,   465, 13476,    11,   339]), tensor([  465, 13476,    11,   339,   550,  5710,   465, 12036]), tensor([  550,  5710,   465, 12036,    11,  6405,   257,  5527]), tensor([   11,  6405,   257,  5527, 27075,    11,   290,  4920]), tensor([27075,    11,   290,  4920,  2241,   287,   257,  4489]), tensor([ 2241,   287,   257,  4489,    64,   319,   262, 34686]), tensor([   64,   319,   262, 34686, 41976,    13,   357, 10915]), tensor([41976,    13,   357, 10915,   314,  2138,

Notes: Print a quick view of `input_ids` and `target_ids` stored in the dataset object to verify sliding-window construction and tensor shapes.

In [31]:
#3 drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training.
# overlap could lead to increased overfitting.
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")                         #1
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)   #2
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,     #3
        num_workers=num_workers     #4
    )

    return dataloader

Notes: `create_dataloader_v1` wraps the dataset into a `DataLoader`, exposing parameters like `batch_size`, `max_length`, `stride`, and `drop_last`. Use this to produce batched training data with optional shuffling and parallel workers.

In [40]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=8, stride=2, shuffle=False)
data_iter = iter(dataloader)      #1
first_batch = next(data_iter)
print(first_batch)

[tensor([  367,  2885,  1464,  1807,  3619,   402,   271, 10899]), tensor([ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]), tensor([ 3619,   402,   271, 10899,  2138,   257,  7026, 15632]), tensor([  271, 10899,  2138,   257,  7026, 15632,   438,  2016]), tensor([ 2138,   257,  7026, 15632,   438,  2016,   257,   922]), tensor([ 7026, 15632,   438,  2016,   257,   922,  5891,  1576]), tensor([ 438, 2016,  257,  922, 5891, 1576,  438,  568]), tensor([ 257,  922, 5891, 1576,  438,  568,  340,  373]), tensor([5891, 1576,  438,  568,  340,  373,  645, 1049]), tensor([ 438,  568,  340,  373,  645, 1049, 5975,  284]), tensor([ 340,  373,  645, 1049, 5975,  284,  502,  284]), tensor([ 645, 1049, 5975,  284,  502,  284, 3285,  326]), tensor([5975,  284,  502,  284, 3285,  326,   11,  287]), tensor([ 502,  284, 3285,  326,   11,  287,  262, 6001]), tensor([3285,  326,   11,  287,  262, 6001,  286,  465]), tensor([   11,   287,   262,  6001,   286,   465, 13476,    11]), tensor([  262, 

Notes: Re-open `the-verdict.txt` (again) and build a `DataLoader` instance for a small `batch_size`. Fetch the first batch via `iter(dataloader)` and `next()` to inspect a sample batch.

In [41]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4,
    shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

[tensor([ 367, 2885, 1464, 1807]), tensor([ 3619,   402,   271, 10899]), tensor([ 2138,   257,  7026, 15632]), tensor([ 438, 2016,  257,  922]), tensor([5891, 1576,  438,  568]), tensor([ 340,  373,  645, 1049]), tensor([5975,  284,  502,  284]), tensor([3285,  326,   11,  287]), tensor([ 262, 6001,  286,  465]), tensor([13476,    11,   339,   550]), tensor([ 5710,   465, 12036,    11]), tensor([ 6405,   257,  5527, 27075]), tensor([  11,  290, 4920, 2241]), tensor([ 287,  257, 4489,   64]), tensor([  319,   262, 34686, 41976]), tensor([   13,   357, 10915,   314]), tensor([2138, 1807,  340,  561]), tensor([  423,   587, 10598,   393]), tensor([28537,  2014,   198,   198]), tensor([   1,  464, 6001,  286]), tensor([  465, 13476,     1,   438]), tensor([5562,  373,  644,  262]), tensor([1466, 1444,  340,   13]), tensor([ 314,  460, 3285, 9074]), tensor([   13, 46606,   536,  5469]), tensor([  438, 14363,   938,  4842]), tensor([1650,  353,  438, 2934]), tensor([  489,  3255,   465, 4842

Notes: Create a larger batched `DataLoader` and print the `inputs` and `targets` for a single batch to visualize batch shapes and example token sequences.

In [56]:
vocab_size = 6
output_dim = 3

Notes: Define small toy vocabulary and embedding dimensions (`vocab_size=6`, `output_dim=3`) to illustrate embedding lookups without the full large GPT vocabulary.

In [57]:
torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


Notes: Initialize a small `Embedding` layer with a fixed manual seed for deterministic weights. This is useful for debugging and inspecting embedding tensors.

In [58]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


Notes: Show the embedding vector for a single token id (`3`) using the small toy embedding layer; useful for inspecting individual token representations.

In [59]:
input_ids = torch.tensor([2, 3, 5, 1])

Notes: Define an example token sequence (`input_ids`) as a 1-D tensor to feed into embedding lookups or the model.

In [60]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


Notes: Look up embeddings for the example token sequence defined above. The output shape should match `[sequence_length, output_dim]`.

In [62]:
vocab_size = 50257
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

Notes: Create a full-sized token embedding layer suitable for GPT-style models using the standard GPT-2 vocabulary size (`50257`) and a larger embedding dimension (`256`). This simulates realistic model dimensions.

In [63]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


Notes: Map the `inputs` batch through the `token_embedding_layer` to produce per-token embeddings for the batch; printing the shape helps verify batch and embedding dimensions.

In [42]:
print(torch.backends.mps.is_available())

True


Notes: Quick runtime check to see if Apple's Metal Performance Shaders (MPS) backend is available on this machine (`torch.backends.mps.is_available()`), useful for macOS GPU acceleration.

## 2 layer perceptron

Notes: This section introduces a small 2-layer perceptron (MLP) used in later demonstration cells. The MLP is kept intentionally simple for clarity.

In [44]:
# prerequisite
# A multilayer perceptron with two hidden layers
# A perceptron is basically a small mathematical decision-making unit.
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):    #1
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),    #2
            torch.nn.ReLU(),               #3

            # 2nd hidden layer
            torch.nn.Linear(30, 20),    #4
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits           #5

Notes: Defines a `NeuralNetwork` class implementing a 2-hidden-layer MLP using `torch.nn.Sequential`. This serves as a simple example model to demonstrate forward passes and parameter inspection.

In [45]:
model = NeuralNetwork(50, 3)


Notes: Instantiate the `NeuralNetwork` with `num_inputs=50` and `num_outputs=3`. This prepares the model for weight inspection and test forward passes.

In [46]:
print(model)


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Notes: Print the model architecture to see the sequential layers and confirm parameter shapes for each layer.

In [47]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


Notes: Compute and print the total number of trainable parameters in the model. Useful to understand model capacity and compare architectures.

In [48]:
print(model.layers[0].weight)


Parameter containing:
tensor([[-0.1072,  0.0573, -0.0647,  ...,  0.1134,  0.0836,  0.1023],
        [ 0.1379, -0.0788, -0.1247,  ..., -0.0639,  0.1120, -0.1065],
        [-0.1353,  0.0778,  0.0809,  ...,  0.0095, -0.0443, -0.1146],
        ...,
        [ 0.0320,  0.1255,  0.0559,  ..., -0.1293, -0.1335,  0.0929],
        [ 0.0115, -0.0670, -0.1291,  ..., -0.1271,  0.1331, -0.0320],
        [-0.0375, -0.1165,  0.0699,  ..., -0.0258, -0.1345,  0.0904]],
       requires_grad=True)


Notes: Print the weight tensor of the model's first Linear layer to inspect raw parameter values and confirm initialization.

In [49]:
print(model.layers[0].weight.shape)


torch.Size([30, 50])


Notes: Print the shape of the first layer's weight tensor to confirm the expected dimensions (should match `[out_features, in_features]`).

In [50]:
model.layers[0].bias

Parameter containing:
tensor([-0.1097,  0.1000, -0.1189, -0.1152,  0.0345, -0.0895,  0.0232,  0.1182,
         0.1249,  0.0121,  0.1185,  0.0552, -0.0991,  0.1094,  0.1274,  0.0516,
         0.0530,  0.0002, -0.0777, -0.0253,  0.0731,  0.1385, -0.0069, -0.0555,
         0.0072, -0.0015,  0.1128, -0.0380,  0.0256, -0.1325],
       requires_grad=True)

Notes: Access the bias tensor for the first layer. If you want to inspect values, wrap this access in a `print(...)` statement — currently the cell evaluates the tensor object.

In [51]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


Notes: Re-seed the RNG and re-instantiate the model to show deterministic initialization. Print the first layer weights to observe the deterministic values.

In [52]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


Notes: Create a random input `X` and run it through the model to produce raw logits. This demonstrates a forward pass and the kind of output dimensionality to expect.

In [53]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


Notes: Compute softmax probabilities from the model logits using `torch.no_grad()` for inference. This simulates model prediction without tracking gradients.